# Logits Extractor Module

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': True,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'base', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks/mitigation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        }
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/log.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [6]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### GPU

In [7]:
! nvidia-smi

Tue Jul  1 19:59:34 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 12.3     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-PCI...  Off  | 00000000:61:00.0 Off |                    0 |
| N/A   34C    P0    34W / 250W |      4MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [8]:
torch.__version__

'2.1.2+cu121'

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [10]:
torch.cuda.memory_allocated()

0

## Logits Extractor
>
> Extracting Tensor Logits from a given Neural Code Model
>

#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

2025-07-01 19:59:35.619873: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751399975.636181  865251 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751399975.641826  865251 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-01 19:59:35.660925: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "float32",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size": 32016
}

In [14]:
model.to(device) #WARNING, Verify the device before assigning to memory

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32016, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_e

#### Dataset

In [15]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['current']}.json")

In [16]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
1800,204481,9c19aff7c7561e3a82978a272ecdaad40dda5c00,django,django/core/files/locks.py,locks.py,_fd,Refs #33476 -- Reformatted code with Black.,def _fd(f):\n \n return f.fileno() if ha...,https://github.com/django/django.git,Python,...,161,26,W0611,18,4,18,51,"from ctypes.wintypes import BOOL, DWORD, HANDLE",Warning,297
2439,23921,bbca1e0d66298fd21abcb953a4c73379b8e04996,PaddleOCR,ppocr/postprocess/__init__.py,__init__.py,build_post_process,add pr,"def build_post_process(config, global_config=N...",https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,240,16,W0611,12,8,12,51,from .pse_postprocess import PSEPostProcess,Warning,339
3069,25049,b5268dc3a0847dce2668265e07ff50d54265b2d8,PaddleOCR,ppocr/modeling/necks/__init__.py,__init__.py,build_neck,add centripetal text model,def build_neck(config):\n from .db_fpn impo...,https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,241,33,W0611,1,4,1,44,"from .db_fpn import DBFPN, RSEFPN, LKPAN",Warning,304
3158,13092,e932d6f9e042b06771fde2a6bec8f256c223a4d6,jina,tests/integration/hub_usage/test_hub_usage.py,test_hub_usage.py,test_use_from_local_hub_deployment_level,fix(hubio.fetch_meta): move significant params...,def test_use_from_local_hub_deployment_level(m...,https://github.com/jina-ai/jina.git,Python,...,39,11,W0611,1,4,1,52,"from jina.hubble.hubio import HubExecutor, HubIO",Warning,63
6459,46920,fb59a76f2d7e3544ad5fc109db678bf5ce5b8f01,airflow,tests/serialization/test_dag_serialization.py,test_dag_serialization.py,test_serialize_and_deserialize_custom_ti_deps,Support dag serialization with custom ti_deps ...,def test_serialize_and_deserialize_custom_ti_d...,https://github.com/apache/airflow.git,Python,...,16,4,W0611,1,8,1,53,from test_plugin import CustomTestTriggerRule,Warning,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458070,25210,1f9400dd7374ce9cc47981372e324ff412e53ba3,PaddleOCR,ppocr/modeling/necks/__init__.py,__init__.py,build_neck,add drrg,def build_neck(config):\n from .db_fpn impo...,https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,255,35,W0611,11,4,11,29,from .ct_fpn import CTFPN,Warning,328
460116,124990,3a345a470c68c71a2a79d38245db25b4d52a2e62,ray,doc/source/ray-air/doc_code/predictors.py,predictors.py,calculate_accuracy,[AIR/Docs] Add Predictor Docs (#25833),def calculate_accuracy(df):\n return pd.Dat...,https://github.com/ray-project/ray.git,Python,...,300,36,W0611,21,0,21,41,from ray.train.predictor import Predictor,Warning,458
461557,25317,f3f473d3f9b91e06d3c0122eff10e03030729d13,PaddleOCR,ppocr/modeling/heads/__init__.py,__init__.py,build_head,update CAN model,def build_head(config):\n # det head\n f...,https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,449,62,W0611,13,4,13,43,from .rec_att_head import AttentionHead,Warning,628
461719,145516,372c620f58c2269dccd5a871a72aebb9df76e32c,ray,doc/source/tune/doc_code/pytorch_optuna.py,pytorch_optuna.py,forward,[docs] Tune overhaul part II (#22656)\n\nCo-au...,"def forward(self, x):\n x = F.relu(F.ma...",https://github.com/ray-project/ray.git,Python,...,115,17,W0611,9,0,9,12,import torch,Warning,153


In [17]:
#df_dataset = df_dataset[df_dataset['input_lenght']>=700]
#df_dataset = df_dataset[:20]

#### Logit Inference

In [18]:
def logit_extractor(model, batch, tf_encoded_inputs, from_index=0):
    """
    Output is the class CausalLMOutputWithPast (https://huggingface.co/transformers/v4.10.1/main_classes/output.html?highlight=causallmoutputwithpast)"
    logits (torch.FloatTensor of shape (batch_size, sequence_length, config.vocab_size)) – Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
    The expression i.type(torch.LongTensor).to(device) is for casting labels for the loss
    """
    callbacks_dir = f"{params['callbacks_dir']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
    create_folder(callbacks_dir)
    
    for idx, n in enumerate(range(from_index, len(tf_encoded_inputs), batch)):
        torch.cuda.empty_cache()
        output = []
        for encoded_sample in tf_encoded_inputs[n:n+batch]:
            output.append( 
                model(input_ids = encoded_sample, labels = encoded_sample)
            )
        output_logits = [ o['logits'].detach().to('cpu').numpy() for o in output ]  #Logits Extraction
        output_loss = np.array([ o.loss.detach().to('cpu').numpy() for o in output ])  #Language modeling loss (for next-token prediction).

        #Saving Callbacks
        current_batch = idx + (from_index//batch)
        for jdx, o_logits in enumerate(output_logits):
            np.save(f"{callbacks_dir}/logits_tensor[{jdx+n}]_batch[{current_batch}].npy", o_logits)
        np.save(f"{callbacks_dir}/_loss_batch[{current_batch}].npy", output_loss)
        
        print(f"Batch [{current_batch}] Completed")

        #Memory Released
        for out in output:
            del out.logits
            torch.cuda.empty_cache()
            del out.loss
            torch.cuda.empty_cache()
        for out in output_logits:
            del out
            torch.cuda.empty_cache()
        for out in output_loss:
            del out
            torch.cuda.empty_cache()

In [19]:
#Casting Integers to Tensor Integers. Make sure the tesor is created in a device
#We ignored the parameter attention_mask since we are not using masking here [https://huggingface.co/transformers/v4.10.1/glossary.html#attention-mask]
tf_encoded_inputs = [tokenizer(sample, return_tensors='pt')['input_ids'].to(device) for sample in df_dataset[params['dataset']['content_column']].values]

In [20]:
## ACTUAL EXPERIMENT
## TIME AND MEMORY CONSUMING
logit_extractor(
    model = model,
    batch = 1, 
    tf_encoded_inputs = tf_encoded_inputs, 
    from_index=0
)

Batch [0] Completed
Batch [1] Completed
Batch [2] Completed
Batch [3] Completed
Batch [4] Completed
Batch [5] Completed
Batch [6] Completed
Batch [7] Completed
Batch [8] Completed


KeyboardInterrupt: 

In [22]:
print("================================= PROCESS COMPLETE =================================")

================================= PROCESS COMPLETE =================================


In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()

: 